# UniSweet raw/P&L-aligned report visuals

**Audience:** UniSweet Leadership and Finance Business Partners  
**Use:** English, annotation-rich components for distributed executive slides  
**Big Idea:** The category barely moved but we did - Mainstream, and Modern Trade above all, gave back the value; PBO survived only because marketing and supply chain spend fell faster than Gross Profit, a lever that cannot be pulled twice.

Internal Sales and P&L compare **FY2024 with FY2023**. Market data is a separate **MAT Nov'24 versus MAT-1** view. All Sales rows are retained so the Sales headlines reconcile exactly to P&L; rows flagged `TURNOVER_GT_GSV` remain subject to business validation.

In [1]:
from pathlib import Path
import importlib.util
import os
import sys
import textwrap
from xml.etree import ElementTree

os.environ.setdefault("MPLCONFIGDIR", "/tmp/unisweet-mpl-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/unisweet-xdg-cache")

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, Rectangle
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts" / "storyline_metrics.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate scripts/storyline_metrics.py")


PROJECT_ROOT = find_project_root()
METRICS_PATH = PROJECT_ROOT / "scripts" / "storyline_metrics.py"
spec = importlib.util.spec_from_file_location("storyline_metrics", METRICS_PATH)
storyline_metrics = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(storyline_metrics)
D = storyline_metrics.STORYLINE_DATAFRAMES

# The storytelling-with-data skill is the source of truth for the visual
# system. It is imported here, not in the appendix, so save_component() can
# lint every report figure as it is exported. Both modules force the Agg
# backend on import, so hand the notebook's own backend back afterwards.
SKILL_DIR = PROJECT_ROOT / ".claude" / "skills" / "storytelling-with-data"
for path in (str(SKILL_DIR), str(PROJECT_ROOT / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

_NOTEBOOK_BACKEND = matplotlib.get_backend()
import lint
import swd
matplotlib.use(_NOTEBOOK_BACKEND, force=True)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "report_visuals"
DATA_DIR = OUTPUT_DIR / "data"
# Headerless twins for the slide deck: same canvas, same axes geometry, with
# the figure-level header and source line hidden so native slide text can sit
# in the bands they leave behind.
SLIDE_DIR = OUTPUT_DIR / "slides"
for _d in (OUTPUT_DIR, DATA_DIR, SLIDE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Muted two-ramp visual system, shared with swd.SEQ_BLUE / swd.SEQ_ORANGE.
# BLUE MARKS FAVOURABLE MOVEMENT, ORANGE MARKS ADVERSE MOVEMENT. Favourable
# means good for UniSweet, which is not always a positive number: a €15.5m cut
# in marketing spend is blue. Direction is never left to colour alone - the
# printed sign says it, and a hatch marks every bar whose value is positive.
PRIMARY = "#004c6d"       # totals and anchors, headline, panel titles
FOCUS = "#346888"
SECONDARY = "#5886a5"     # favourable movement
SUPPORT = "#7aa6c2"
CONTEXT = "#9dc6e0"       # fills for favourable data that is context
PALE = "#c1e7ff"          # favourable washes and card fills

CONTRA = "#f79545"        # adverse movement - the losses that are the story
CONTRA_MID = "#faa35e"
CONTRA_SOFT = "#fdb177"   # adverse, but context rather than signal
CONTRA_PALE = "#ffdbc1"   # adverse washes and card fills

# No step of the orange ramp reaches 4.5:1 against white or against its own
# pale tint (#f79545 manages 2.3:1), so orange is a fill and mark colour only.
# Text that sits on or beside an orange fill is INK.

# Grey is what everything that is NOT the story is made of. Only data the
# headline actually argues about gets a hue; anchors, sums, offsets and
# not-this-half context recede into the page. Two greys because fills and text
# need different weights: a #8c8c8c fill reads as background, but #8c8c8c text
# is only 3.1:1 on white, so muted TEXT is MID.
INK = "#262626"           # text that must be read
MID = "#595959"           # secondary and de-emphasised TEXT
BASE = "#8c8c8c"          # context data: present, not competing
MUTED = "#bfbfbf"         # totals and anchors; zero lines; the source line
RULE = "#d9d9d9"          # the one remaining spine
WHITE = "#ffffff"

ARIAL_PATH = font_manager.findfont("Arial", fallback_to_default=False)
plt.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "figure.facecolor": WHITE,
    "axes.facecolor": WHITE,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "xtick.color": MID,
    "ytick.color": MID,
    "axes.edgecolor": RULE,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "svg.fonttype": "none",
})

MANIFEST = []


def new_figure(*, nrows=1, ncols=1, gridspec_kw=None):
    return plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(40 / 3, 7.5),
        gridspec_kw=gridspec_kw,
    )


def _chrome(fig, artist):
    """Register figure-level furniture so the slide export can hide it.

    Only the header and source line are chrome. In-axes callouts and the
    caption texts in charts 01 and 08 are the annotation layer a reader
    depends on when no presenter is in the room, so they are never hidden.
    """
    fig._chrome = getattr(fig, "_chrome", []) + [artist]
    return artist


def add_header(fig, title, subtitle):
    # The headline owns the top-left corner, where the eye enters. Titles are
    # edited to fit; they never shrink to rescue overlong copy.
    _chrome(fig, fig.suptitle(title, x=0.055, y=0.965, ha="left", fontsize=23.0,
                              fontweight="bold", color=PRIMARY))
    _chrome(fig, fig.text(0.055, 0.915, subtitle, ha="left", va="top", fontsize=11.5, color=MID))


def add_footer(fig, source, caveat=""):
    footer = f"Source: {source}"
    if caveat:
        footer += f"  |  {caveat}"
    # Present for provenance, never competing: the source line is the faintest
    # thing on the page.
    _chrome(fig, fig.text(0.055, 0.025, footer, ha="left", va="bottom", fontsize=8.5, color=MUTED))


def add_callout(ax, x, y, text, color=PRIMARY, fontsize=13, **kwargs):
    """Narrative text, always read from a left edge (the Z path).

    x/y are axes fractions, so a callout keeps its place when limits change.
    Labels attached to a mark are the exception and stay on their mark.
    """
    return ax.text(x, y, text, transform=ax.transAxes, ha="left", va="top",
                   fontsize=fontsize, color=color, fontweight="bold", **kwargs)


def clean_axis(ax, baseline=False):
    """No gridlines, no border, no tick marks.

    Once every mark carries a direct label the value axis goes with the
    gridlines - a tick that repeats a printed number is the same clutter.
    `baseline=True` keeps the bottom spine for the waterfalls, whose bars are
    anchored on it; every other chart draws its own explicit zero line.
    """
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_visible(baseline)
    if baseline:
        ax.spines["bottom"].set_color(RULE)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)


def signed(value, decimals=1, suffix=""):
    return f"{value:+,.{decimals}f}{suffix}"


def save_component(visual_id, page, title, fig, data, source, caveat=""):
    data_path = DATA_DIR / f"{visual_id}.csv"
    png_path = OUTPUT_DIR / f"{visual_id}.png"
    svg_path = OUTPUT_DIR / f"{visual_id}.svg"
    data.to_csv(data_path, index=False)
    fig.savefig(png_path, dpi=144, facecolor=WHITE)
    fig.savefig(svg_path, facecolor=WHITE)

    # Slide twin. The axes geometry is deliberately untouched: hiding the
    # chrome leaves empty bands exactly where add_header and add_footer drew,
    # and the deck puts native text back at the same coordinates. The chart
    # keeps the alignment it was tuned for and nothing reflows.
    chrome = getattr(fig, "_chrome", [])
    for artist in chrome:
        artist.set_visible(False)
    fig.savefig(SLIDE_DIR / f"{visual_id}.png", dpi=144, facecolor=WHITE)
    for artist in chrome:
        artist.set_visible(True)
    # Advisory, never fatal: chart 03 is a contribution bridge drawn from a
    # 58.0 pp baseline, so ZERO_BASELINE fires there by design. That truncation
    # is stated in the chart's own caveat line rather than silently redrawn.
    findings = lint.check(fig)
    for finding in findings:
        print(f"lint {visual_id}: {finding}")
    MANIFEST.append({
        "visual_id": visual_id,
        "storyline_page": page,
        "action_title": title,
        "png": png_path.relative_to(PROJECT_ROOT).as_posix(),
        "svg": svg_path.relative_to(PROJECT_ROOT).as_posix(),
        "data_csv": data_path.relative_to(PROJECT_ROOT).as_posix(),
        "source": source,
        "caveat": caveat,
        "lint": "; ".join(str(f) for f in findings) or "clean",
    })
    plt.show()


def draw_segment_panel(ax, frame, story):
    """Two bars a row - MAT-1 then MAT - with the growth rate in its own column.

    `story` names the one segment the action title argues about. Every other row
    is backdrop and goes grey: mark, tick label and value label together. Greying
    the bar but leaving a bold black number beside it pulls the eye straight back
    to the row you just muted.

    De-emphasised TEXT is MID, never BASE or MUTED. A #8c8c8c fill reads as
    background, but #8c8c8c text is only 3.1:1 on white and #bfbfbf is 2.3:1;
    backdrop rows recede on weight and size, not by becoming unreadable.
    """
    y = np.arange(len(frame))[::-1].astype(float)
    height = 0.34
    span = float(frame["mat_meur"].max())

    for i, row in enumerate(frame.itertuples(index=False)):
        top = y[i]
        is_story = row.segment == story
        current = (SECONDARY if row.growth_pct >= 0 else CONTRA) if is_story else BASE
        ax.barh(top + height / 2 + 0.02, row.mat1_meur, height=height,
                color=MUTED, edgecolor=RULE, linewidth=0.6)
        bar = ax.barh(top - height / 2 - 0.02, row.mat_meur, height=height,
                      color=current, edgecolor=PRIMARY if is_story else RULE,
                      linewidth=0.6)
        if is_story and row.growth_pct >= 0:
            bar[0].set_hatch("///")

        ax.text(row.mat1_meur + span * 0.012, top + height / 2 + 0.02,
                f"{row.mat1_meur:,.1f}", va="center", ha="left", fontsize=10, color=MID)
        ax.text(row.mat_meur + span * 0.012, top - height / 2 - 0.02,
                f"{row.mat_meur:,.1f}", va="center", ha="left",
                fontsize=10.5 if is_story else 10,
                color=INK if is_story else MID,
                fontweight="bold" if is_story else "normal")
        ax.text(1.005, top, f"{row.growth_pct:+.1%}", transform=ax.get_yaxis_transform(),
                va="center", ha="left", fontsize=15 if is_story else 12.5,
                color=INK if is_story else MID,
                fontweight="bold" if is_story else "normal")

    ax.set_yticks(y, frame["segment"].tolist(), fontsize=13)
    for tick, segment in zip(ax.get_yticklabels(), frame["segment"]):
        tick.set_color(INK if segment == story else MID)
        tick.set_fontweight("bold" if segment == story else "normal")
    ax.set_xticks([])
    ax.set_xlim(0, span * 1.13)
    ax.set_ylim(-0.70, len(frame) - 0.42)
    # Category is the file's own total, not the sum of the rows beneath it.
    ax.axhline(y[0] - 0.5, color=RULE, linewidth=1)
    clean_axis(ax)

    ax.text(1.005, len(frame) - 0.60, "MAT growth", transform=ax.get_yaxis_transform(),
            va="center", ha="left", fontsize=10, color=MID, fontweight="bold")
    ax.text(0.0, len(frame) - 0.60, "Paler bar: MAT-1    Solid bar: MAT",
            transform=ax.get_yaxis_transform(), va="center", ha="left",
            fontsize=10, color=MID)


def draw_paired_brands(ax, brands, prior, current, panel_title, story):
    """Prior versus current cost by brand, with only `story` brands in hue.

    COBALT is deliberately not a story brand on the cost page. Its spend rose
    EUR0.5m, which drawn in the adverse hue became the loudest mark on a page
    whose whole claim is that spend came down - and spending more behind a
    growing brand is not adverse anyway.
    """
    y = np.arange(len(brands))[::-1].astype(float)
    height = 0.34
    span = float(max(prior.max(), current.max()))
    for i, brand in enumerate(brands):
        top = y[i]
        change = current[i] - prior[i]
        is_story = brand in story
        # Favourable means good for UniSweet, not simply a negative number: a cost
        # that falls is blue even though it prints with a minus sign.
        colour = (SECONDARY if change <= 0 else CONTRA) if is_story else BASE
        ax.barh(top + height / 2 + 0.02, prior[i], height=height,
                color=MUTED, edgecolor=RULE, linewidth=0.6)
        bar = ax.barh(top - height / 2 - 0.02, current[i], height=height,
                      color=colour, edgecolor=PRIMARY if is_story else RULE, linewidth=0.6)
        if is_story and change > 0:
            bar[0].set_hatch("///")
        ax.text(prior[i] + span * 0.015, top + height / 2 + 0.02, f"{prior[i]:,.1f}",
                va="center", ha="left", fontsize=9.5, color=MID)
        ax.text(current[i] + span * 0.015, top - height / 2 - 0.02,
                f"{current[i]:,.1f}   ({change / prior[i]:+.0%})", va="center", ha="left",
                fontsize=10.5 if is_story else 10,
                color=INK if is_story else MID,
                fontweight="bold" if is_story else "normal")
    ax.set_yticks(y, list(brands), fontsize=12)
    for tick, brand in zip(ax.get_yticklabels(), brands):
        tick.set_fontweight("bold" if brand in story else "normal")
        tick.set_color(INK if brand in story else MID)
    ax.set_xticks([])
    ax.set_xlim(0, span * 1.34)
    ax.set_ylim(-0.75, len(brands) - 0.25)
    clean_axis(ax)
    ax.set_title(panel_title, loc="left", fontsize=14, color=PRIMARY,
                 fontweight="bold", pad=14)


print(f"Arial font: {ARIAL_PATH}")
print(f"Raw storyline metrics: {METRICS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Visual system: swd.ACCENT {swd.ACCENT} / swd.ACCENT_2 {swd.ACCENT_2}")


Arial font: /System/Library/Fonts/Supplemental/Arial.ttf
Raw storyline metrics: scripts/storyline_metrics.py
Visual system: swd.ACCENT #004c6d / swd.ACCENT_2 #f79545


## Validation before charting

In [2]:
reconciliation = D["pnl_reconciliation"]
check_columns = [column for column in reconciliation if column.startswith("check_")]
assert np.allclose(reconciliation[check_columns], 0.0, atol=1e-8)

topline_bridge = D["topline_bridge"]
assert np.isclose(
    topline_bridge.loc[0, "value_keur"] + topline_bridge.loc[1:2, "value_keur"].sum(),
    topline_bridge.loc[3, "value_keur"],
)
share_bridge = D["market_share_bridge"]
assert np.isclose(
    share_bridge.loc[0, "value_pp"] + share_bridge.loc[1:4, "value_pp"].sum(),
    share_bridge.loc[5, "value_pp"],
)
pbo_bridge = D["pbo_bridge"]
assert np.isclose(
    pbo_bridge.loc[0, "value_keur"] + pbo_bridge.loc[1:2, "value_keur"].sum(),
    pbo_bridge.loc[3, "value_keur"],
)

display(reconciliation.round(6))
display(D["quality"])

,reporting_year,gsv_keur,turnover_keur,discount_keur,pnl_gsv_keur,check_gsv_keur,pnl_turnover_keur,check_turnover_keur,pnl_discount_keur,check_discount_keur
0,2023,371528.3,290066.9,81461.4,371528.3,-0.0,290066.9,0.0,81461.4,-0.0
1,2024,349247.6,270504.3,78743.3,349247.6,0.0,270504.3,-0.0,78743.3,0.0


,metric,value
0,Raw Sales rows,6953.0
1,Flagged TO > GSV TO FY2023,2574.5
2,Flagged TO > GSV TO FY2024,3173.0


## Page 1 - Where we ended up


In [3]:
# Three cards, one per line of the brief: TO, GSV and Discount on Turnover.
# All three moved against us, so none of them is tinted blue to make the row
# look balanced - the page says that plainly instead.
CARD_SPEC = [
    ("Turnover", "Turnover", "money"),
    ("Gross Sales Value", "GSV", "money"),
    ("Discount % of Turnover", "Discount / TO", "rate"),
]
metrics = D["headline"].set_index("metric")
cards = pd.DataFrame([
    {
        "metric": label,
        "value_2023": metrics.loc[key, "value_2023"],
        "value_2024": metrics.loc[key, "value_2024"],
        "display": (f"€{metrics.loc[key, 'value_2024'] / 1_000:,.1f}m" if kind == "money"
                    else f"{metrics.loc[key, 'value_2024']:.2%}"),
        "delta": (f"{metrics.loc[key, 'absolute_change'] / 1_000:+,.1f}m" if kind == "money"
                  else f"{metrics.loc[key, 'absolute_change'] * 10_000:+,.0f}bps"),
        "context": (f"{metrics.loc[key, 'growth_pct']:+.1%} YoY" if kind == "money"
                    else f"from {metrics.loc[key, 'value_2023']:.2%}"),
    }
    for label, key, kind in CARD_SPEC
])

fig, axes = new_figure(nrows=1, ncols=3)
fig.subplots_adjust(left=0.05, right=0.97, bottom=0.30, top=0.85, wspace=0.09)
title = "Every topline KPI moved against us in FY2024"
add_header(fig, title, "FY2024 versus FY2023, P&L-aligned reporting base")

for ax, row in zip(axes.flat, cards.itertuples(index=False)):
    ax.set_axis_off()
    ax.add_patch(FancyBboxPatch(
        (0.01, 0.02), 0.98, 0.94, boxstyle="round,pad=0.012,rounding_size=0.03",
        transform=ax.transAxes, facecolor=CONTRA_PALE, edgecolor=CONTRA, linewidth=1.5))
    ax.text(0.07, 0.83, row.metric.upper(), transform=ax.transAxes,
            fontsize=11, color=MID, fontweight="bold")
    # INK, never orange: no step of the orange ramp is readable as text.
    ax.text(0.07, 0.47, row.display, transform=ax.transAxes,
            fontsize=40, color=INK, fontweight="bold")
    ax.text(0.07, 0.20, row.delta, transform=ax.transAxes,
            fontsize=17, color=INK, fontweight="bold")
    ax.text(0.07, 0.09, row.context, transform=ax.transAxes, fontsize=12, color=MID)

fig.text(0.055, 0.20,
         "Turnover fell faster than GSV because discount intensity rose at the same "
         "time - the base shrank and we kept less of what was left.",
         ha="left", va="top", fontsize=14.5, color=PRIMARY, fontweight="bold")

source = "P&L Table.xlsx and outputs/sales_master.csv"
gsv_rate = metrics.loc["Discount / GSV"]
caveat = (f"Discount is stated on Turnover, the P&L convention; on GSV the same movement "
          f"reads {gsv_rate['value_2023']:.2%} to {gsv_rate['value_2024']:.2%} "
          f"({gsv_rate['absolute_change'] * 10_000:+,.0f}bps)")
add_footer(fig, source, caveat)
save_component("01_scorecard_to_gsv_discount", 1, title, fig, cards, source, caveat)


<Figure size 1333.33x750 with 3 Axes>

## Page 2 - The market barely moved


In [4]:
segments = D["market_segment"]
frame = (segments[segments["channel"].eq("Total")]
         .set_index("segment").loc[storyline_metrics.SEGMENT_ROW_ORDER].reset_index())
coverage = float(frame["segment_coverage_pct"].iloc[0])
share_movement = float(frame.loc[frame["segment"].eq("Category"),
                                 "unisweet_share_movement_pp"].iloc[0])
# OLIVE's share of FY2024 Turnover, which is why Mainstream decides the year. The
# source workbook's own 'TO Contribution %' divides by the FY2023 total instead
# and reads 73.3%; this is the FY2024-on-FY2024 number.
brand_pnl = D["pnl"].set_index("brand")
olive_weight = brand_pnl.loc["OLIVE", "turnover_2024_keur"] / brand_pnl.loc["TOTAL", "turnover_2024_keur"]

fig, ax = new_figure()
fig.subplots_adjust(left=0.145, right=0.90, bottom=0.11, top=0.80)
title = "The category barely moved; Mainstream is where the value went"
add_header(fig, title, "Market sales value, EURm  |  MAT Nov'24 versus MAT-1  |  All channels")

draw_segment_panel(ax, frame, story="Mainstream")

# Every number in the callout is read back out of the frame, so the prose and the
# bars cannot round to different figures.
growth = dict(zip(frame["segment"], frame["growth_pct"]))
add_callout(ax, 0.30, 0.36,
            f"The category fell {abs(growth['Category']):.1%}, but our share fell "
            f"{abs(share_movement):.2f} points.\n"
            f"Mainstream - where {olive_weight:.0%} of our turnover sits - "
            f"fell {abs(growth['Mainstream']):.1%}.")

source = "Market Report MAT Nov'24.xlsx"
caveat = (f"Segment rows sum the named brands only, which cover {coverage:.1f}% of "
          "Category; Category is the file's own total, not their sum")
add_footer(fig, source, caveat)
save_component("02_market_growth_total", 2, title, fig, frame, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

## Page 2 - Modern Trade is where it broke


In [5]:
frame = (segments[segments["channel"].eq("MT")]
         .set_index("segment").loc[storyline_metrics.SEGMENT_ROW_ORDER].reset_index())
share_movement = float(frame.loc[frame["segment"].eq("Category"),
                                 "unisweet_share_movement_pp"].iloc[0])
lilac = float(D["channel"].set_index("channel").loc["MT", "competitor_share_movement_pp"])

fig, ax = new_figure()
fig.subplots_adjust(left=0.145, right=0.90, bottom=0.11, top=0.80)
title = "In Modern Trade the same category decline cost us six share points"
add_header(fig, title, "Market sales value, EURm  |  MAT Nov'24 versus MAT-1  |  Modern Trade only")

draw_segment_panel(ax, frame, story="Mainstream")

growth = dict(zip(frame["segment"], frame["growth_pct"]))
add_callout(ax, 0.30, 0.36,
            f"MT Mainstream fell {abs(growth['Mainstream']):.1%} while MT Economy grew "
            f"{growth['Economy']:.1%}.\n"
            f"UniSweet share in MT: {share_movement:+.2f} points; LILAC: {lilac:+.2f}.")

source = "Market Report MAT Nov'24.xlsx"
caveat = ("Category growth is identical (-1.3%) at Total, DT and MT - the source scales "
          "it proportionally, so this row repeats page 2 by construction")
add_footer(fig, source, caveat)
save_component("03_market_growth_mt", 2, title, fig, frame, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

## Page 3 - OLIVE in Modern Trade, month by month


In [6]:
frame = D["olive_mt_monthly"].copy()
prior = frame["gsv_keur_2023"].to_numpy() / 1_000
current = frame["gsv_keur_2024"].to_numpy() / 1_000
h2_gap = frame.loc[frame["half"].eq("H2"), "gsv_change_keur"].sum() / 1_000
fy_gap = frame["gsv_change_keur"].sum() / 1_000

fig, ax = new_figure()
fig.subplots_adjust(left=0.06, right=0.905, bottom=0.14, top=0.80)
title = "OLIVE in MT lost €8.73m of GSV, two thirds of it after June"
add_header(fig, title,
           "OLIVE gross sales value in Modern Trade, EURm per month  |  FY2024 versus FY2023")

x = np.arange(12)
# The shaded gap is the chart's subject: it is the money, drawn as an area rather
# than left for the reader to subtract two lines in their head.
ax.fill_between(x, prior, current, where=current <= prior, interpolate=True,
                color=CONTRA_PALE, alpha=0.85, zorder=1)
ax.fill_between(x, prior, current, where=current > prior, interpolate=True,
                color=PALE, alpha=0.9, zorder=1)
ax.plot(x, prior, color=BASE, linewidth=2.2, zorder=3)
ax.plot(x, current, color=CONTRA, linewidth=3.0, zorder=4)

# Marked and labelled: H2, which the title is about, plus February, the one H1
# month big enough to ask about. The rest of H1 runs on as a smooth line - a
# marker on a month nobody will discuss is a dot the reader has to dismiss.
called_out = [1] + list(range(6, 12))
ax.scatter(x[called_out], current[called_out], s=30, color=CONTRA, zorder=5,
           edgecolor=WHITE, linewidth=0.9)
for i in called_out:
    below = current[i] <= prior[i]
    ax.text(i, current[i] - 0.16 if below else current[i] + 0.13,
            f"{current[i]:,.1f}", ha="center", va="top" if below else "bottom",
            fontsize=10.5, color=INK, fontweight="bold")

# Series labels ride the right-hand end of their own line, so neither one can
# collide with the January data labels.
ax.text(11.28, prior[-1], "FY2023", ha="left", va="center", fontsize=13.5,
        color=MID, fontweight="bold")
ax.text(11.28, current[-1], "FY2024", ha="left", va="center", fontsize=13.5,
        color=INK, fontweight="bold")

ax.axvspan(5.5, 11.4, color=CONTRA_PALE, alpha=0.22, zorder=0)
ax.set_xticks(x, ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], fontsize=11)
for tick, half in zip(ax.get_xticklabels(), frame["half"]):
    tick.set_color(INK if half == "H2" else MID)
ax.set_xlim(-0.5, 12.6)
# Zero-based, even though it costs the lower third of the canvas. The shaded gap
# is the subject of the chart, and a truncated axis would draw it roughly twice
# its true size. The empty band above carries the callout instead.
ax.set_ylim(0, max(prior.max(), current.max()) * 1.42)
clean_axis(ax)

# The value axis comes back because H1 is no longer labelled point by point. Drop
# the axis only when every mark carries its own number; here seven of the twelve
# do not, so the reader needs a scale to recover them from.
ticks = np.arange(0, 7.0, 2.0)
ax.set_yticks(ticks, [f"{t:,.0f}" for t in ticks], fontsize=10.5)
ax.set_ylabel("GSV, EURm", fontsize=11, color=MID, labelpad=10)
ax.spines["left"].set_visible(True)
ax.spines["left"].set_color(RULE)

add_callout(ax, 0.015, 0.99,
            f"H2 (shaded): {h2_gap:+,.2f}m of the {fy_gap:+,.2f}m FY gap\n"
            "November is the only month above FY2023",
            fontsize=14)

source = "outputs/sales_master.csv"
caveat = ("Internal sell-in GSV on the raw reporting base; monthly weakness does not "
          "by itself establish seasonality")
add_footer(fig, source, caveat)
save_component("04_olive_mt_monthly_gsv", 3, title, fig, frame, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

## Page 4 - Why profit survived


In [7]:
BRANDS = ["OLIVE", "COBALT", "SKY"]
costs = (D["pnl"].set_index("brand")
         .loc[BRANDS, ["marketing_2023_keur", "marketing_2024_keur",
                       "supply_chain_2023_keur", "supply_chain_2024_keur"]] / 1_000)
costs = costs.reset_index()

fig, axes = new_figure(nrows=1, ncols=2)
fig.subplots_adjust(left=0.075, right=0.97, bottom=0.30, top=0.76, wspace=0.30)
title = "Profit held because we spent less, not because we sold more"
add_header(fig, title,
           "Marketing expense and total supply chain cost, EURm  |  FY2024 versus FY2023  |  "
           "Paler bar: FY2023  |  The two panels are on different scales")

# OLIVE and SKY are the cuts that made the number; COBALT went the other way by
# EUR0.5m and is backdrop here.
cut = {"OLIVE", "SKY"}
draw_paired_brands(axes[0], BRANDS,
                   costs["marketing_2023_keur"].to_numpy(),
                   costs["marketing_2024_keur"].to_numpy(),
                   "Marketing expense", story=cut)
draw_paired_brands(axes[1], BRANDS,
                   costs["supply_chain_2023_keur"].to_numpy(),
                   costs["supply_chain_2024_keur"].to_numpy(),
                   "Total supply chain cost", story=cut)

fig.text(0.075, 0.215,
         "€15.5m less marketing and €9.3m less supply chain cost more than covered\n"
         "the €10.3m of Gross Profit lost - PBO rose 6.4% on cost alone.",
         ha="left", va="top", fontsize=15, color=PRIMARY, fontweight="bold",
         linespacing=1.45)
fig.text(0.075, 0.095,
         "That lever cannot be pulled twice: supply chain cost per euro of Turnover "
         "worsened 35bps, so the saving is a smaller base, not efficiency.",
         ha="left", va="top", fontsize=12, color=MID)

source = "P&L Table.xlsx"
caveat = ("Marketing and supply chain cost are P&L assumptions stated in the source "
          "workbook, not measured spend")
add_footer(fig, source, caveat)
save_component("05_cost_by_brand", 4, title, fig, costs, source, caveat)


<Figure size 1333.33x750 with 2 Axes>

## Page 4 - Where the growth is


In [8]:
GROWTH_BRANDS = ["COBALT", "SKY"]
frame = (D["pnl"].set_index("brand")
         .loc[GROWTH_BRANDS, ["turnover_2023_keur", "turnover_2024_keur", "turnover_growth_pct"]]
         .join(D["portfolio"].set_index("brand")[["market_value_growth_pct", "share_movement_pp"]])
         .reset_index())

fig, ax = new_figure()
fig.subplots_adjust(left=0.10, right=0.66, bottom=0.24, top=0.74)
title = "COBALT is compounding; SKY is losing sell-in into a market that grew 23.8%"
add_header(fig, title, "Turnover, EURm  |  FY2024 versus FY2023  |  Paler bar: FY2023")

y = np.arange(len(frame))[::-1].astype(float)
height = 0.30
span = float(frame["turnover_2024_keur"].max()) / 1_000

for i, row in enumerate(frame.itertuples(index=False)):
    top = y[i]
    prior = row.turnover_2023_keur / 1_000
    current = row.turnover_2024_keur / 1_000
    colour = SECONDARY if row.turnover_growth_pct >= 0 else CONTRA
    ax.barh(top + height / 2 + 0.03, prior, height=height,
            color=MUTED, edgecolor=RULE, linewidth=0.6)
    bar = ax.barh(top - height / 2 - 0.03, current, height=height,
                  color=colour, edgecolor=PRIMARY, linewidth=0.6)
    if row.turnover_growth_pct >= 0:
        bar[0].set_hatch("///")
    ax.text(prior + span * 0.012, top + height / 2 + 0.03, f"{prior:,.2f}",
            va="center", ha="left", fontsize=10, color=MID)
    ax.text(current + span * 0.012, top - height / 2 - 0.03,
            f"{current:,.2f}   ({row.turnover_growth_pct:+.1%})", va="center",
            ha="left", fontsize=12, color=INK, fontweight="bold")

ax.set_yticks(y, frame["brand"].tolist(), fontsize=15)
for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")
    tick.set_color(INK)
ax.set_xticks([])
ax.set_xlim(0, span * 1.30)
ax.set_ylim(-0.7, len(frame) - 0.3)
clean_axis(ax)

# The diagnosis lives beside the bars, because our own growth only means something
# against what the brand's market did.
for i, row in enumerate(frame.itertuples(index=False)):
    agrees = (row.turnover_growth_pct >= 0) == (row.market_value_growth_pct >= 0)
    fig.text(0.695, 0.585 - i * 0.245,
             f"{row.brand}  |  market {row.market_value_growth_pct:+.1%}, "
             f"share {row.share_movement_pp:+.2f}pp",
             fontsize=11.5, color=MID, fontweight="bold")
    fig.text(0.695, 0.535 - i * 0.245,
             ("Sell-in and market move together.\nScale it - once incremental PBO is proven."
              if agrees else
              "Sell-in and market move apart.\nDiagnose it before funding it."),
             fontsize=13, color=INK, va="top", fontweight="bold")

source = "P&L Table.xlsx and Market Report MAT Nov'24.xlsx"
caveat = ("Internal Turnover is FY sell-in; market value is MAT Nov'24 sell-out - "
          "different periods and different measures")
add_footer(fig, source, caveat)
save_component("06_revenue_cobalt_sky", 4, title, fig, frame, source, caveat)


<Figure size 1333.33x750 with 1 Axes>

## Export manifest and render QA

In [9]:
manifest = pd.DataFrame(MANIFEST)
manifest_path = OUTPUT_DIR / "visual_manifest.csv"
manifest.to_csv(manifest_path, index=False)

expected_visuals = {
    "01_scorecard_to_gsv_discount", "02_market_growth_total", "03_market_growth_mt",
    "04_olive_mt_monthly_gsv", "05_cost_by_brand", "06_revenue_cobalt_sky",
}
assert len(manifest) == 6
assert set(manifest["visual_id"]) == expected_visuals
assert manifest["visual_id"].is_unique
assert manifest["lint"].eq("clean").all(), manifest.loc[manifest["lint"].ne("clean"), ["visual_id", "lint"]]

for row in manifest.itertuples(index=False):
    png = PROJECT_ROOT / row.png
    svg = PROJECT_ROOT / row.svg
    csv = PROJECT_ROOT / row.data_csv
    assert png.exists() and svg.exists() and csv.exists()
    with Image.open(png) as rendered:
        assert rendered.size == (1920, 1080), (row.visual_id, rendered.size)
    ElementTree.parse(svg)
    assert "Arial" in svg.read_text(encoding="utf-8")

quality_rubric = pd.DataFrame({
    "dimension": ["Audience", "Action", "Chart fit", "Accuracy", "Clutter", "Attention", "Accessibility", "Story", "Medium fit", "Polish"],
    "score": [2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
    "evidence": [
        "Leadership and FBP audience stated",
        "Action titles and decision implications",
        "Big numbers, paired bars and a two-series line",
        "Raw/P&L reconciliation and explicit period caveats",
        "No gridlines; the one value axis kept is the one with unlabelled points",
        "Grey for context; hue only where the headline argues",
        "Signs, position, labels and hatches supplement colour",
        "Six exhibits, one line of argument, market to action",
        "Annotations and sources support distributed slides",
        "Arial, one left edge for all narrative text, stable 16:9 exports",
    ],
})
quality_rubric.to_csv(OUTPUT_DIR / "quality_rubric.csv", index=False)
assert quality_rubric["score"].sum() >= 17

print(f"Exported {len(manifest)} visual components to {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Quality rubric: {quality_rubric['score'].sum()}/20")
display(manifest)
display(quality_rubric)


Exported 6 visual components to outputs/report_visuals
Quality rubric: 20/20


,visual_id,storyline_page,action_title,png,svg,data_csv,source,caveat,lint
0,01_scorecard_to_gsv_discount,1,Every topline KPI moved against us in FY2024,outputs/report_visuals/01_scorecard_to_gsv_dis...,outputs/report_visuals/01_scorecard_to_gsv_dis...,outputs/report_visuals/data/01_scorecard_to_gs...,P&L Table.xlsx and outputs/sales_master.csv,"Discount is stated on Turnover, the P&L conven...",clean
1,02_market_growth_total,2,The category barely moved; Mainstream is where...,outputs/report_visuals/02_market_growth_total.png,outputs/report_visuals/02_market_growth_total.svg,outputs/report_visuals/data/02_market_growth_t...,Market Report MAT Nov'24.xlsx,"Segment rows sum the named brands only, which ...",clean
2,03_market_growth_mt,2,In Modern Trade the same category decline cost...,outputs/report_visuals/03_market_growth_mt.png,outputs/report_visuals/03_market_growth_mt.svg,outputs/report_visuals/data/03_market_growth_m...,Market Report MAT Nov'24.xlsx,"Category growth is identical (-1.3%) at Total,...",clean
3,04_olive_mt_monthly_gsv,3,"OLIVE in MT lost €8.73m of GSV, two thirds of ...",outputs/report_visuals/04_olive_mt_monthly_gsv...,outputs/report_visuals/04_olive_mt_monthly_gsv...,outputs/report_visuals/data/04_olive_mt_monthl...,outputs/sales_master.csv,Internal sell-in GSV on the raw reporting base...,clean
4,05_cost_by_brand,4,"Profit held because we spent less, not because...",outputs/report_visuals/05_cost_by_brand.png,outputs/report_visuals/05_cost_by_brand.svg,outputs/report_visuals/data/05_cost_by_brand.csv,P&L Table.xlsx,Marketing and supply chain cost are P&L assump...,clean
5,06_revenue_cobalt_sky,4,COBALT is compounding; SKY is losing sell-in i...,outputs/report_visuals/06_revenue_cobalt_sky.png,outputs/report_visuals/06_revenue_cobalt_sky.svg,outputs/report_visuals/data/06_revenue_cobalt_...,P&L Table.xlsx and Market Report MAT Nov'24.xlsx,Internal Turnover is FY sell-in; market value ...,clean


,dimension,score,evidence
0,Audience,2,Leadership and FBP audience stated
1,Action,2,Action titles and decision implications
2,Chart fit,2,"Big numbers, paired bars and a two-series line"
3,Accuracy,2,Raw/P&L reconciliation and explicit period cav...
4,Clutter,2,No gridlines; the one value axis kept is the o...
5,Attention,2,Grey for context; hue only where the headline ...
6,Accessibility,2,"Signs, position, labels and hatches supplement..."
7,Story,2,"Six exhibits, one line of argument, market to ..."
8,Medium fit,2,Annotations and sources support distributed sl...
9,Polish,2,"Arial, one left edge for all narrative text, s..."


## Appendix - Verification of `scripts/metrics.py` against the framework

`scripts/metrics.py` is the **certified** analytical base (`certified_for_analysis = True`).
Section 2.2 of `STORYLINE_METRIC_FRAMEWORK.md` keeps it deliberately separate from the
raw/P&L-aligned base that the storyline headlines use, so the two are *expected* to differ.

This appendix does two things:

1. Confirms every quantified framework claim reconciles on the base it is assigned to.
2. Charts where the certified base would change the storyline, so the sensitivity is visible
   rather than buried.

Charts use the `storytelling-with-data` skill (`.claude/skills/storytelling-with-data/`) and are
linted before export. Exports go to `outputs/report_visuals/verification/` so the certified
11-component manifest above is untouched.

> Running these cells calls `swd.use()`, which replaces the global matplotlib rcParams. The
> project palette is snapshotted below and restored in the final cell, so earlier cells can be
> re-run in any order.

In [10]:
# --- Appendix setup: certified-base verification -------------------------
# swd, lint and sys.path are already installed by the theme cell at the top.
import metrics as CERTIFIED

_PROJECT_RCPARAMS = plt.rcParams.copy()   # restored in the final appendix cell
swd.use()

VERIFY_DIR = OUTPUT_DIR / "verification"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)
VERIFY_MANIFEST = []

FY_PRIOR, FY_CURRENT = CERTIFIED.PRIOR_YEAR, CERTIFIED.CURRENT_YEAR
raw_rows = CERTIFIED.sales_df.assign(y=CERTIFIED.sales_df["reporting_month"].dt.year)
cert_rows = raw_rows[raw_rows["certified_for_analysis"]]


def _to(frame, year, col=None, key=None):
    subset = frame if col is None else frame[frame[col] == key]
    return subset.loc[subset["y"] == year, "turnover_keur"].sum()


def to_change(frame, col=None, key=None):
    return _to(frame, FY_CURRENT, col, key) - _to(frame, FY_PRIOR, col, key)


def to_growth(frame, col=None, key=None):
    prior = _to(frame, FY_PRIOR, col, key)
    return (_to(frame, FY_CURRENT, col, key) / prior - 1) * 100 if prior else float("nan")


def save_verification(fig, visual_id, headline, caveat):
    """Lint, then export PNG + SVG + register in the appendix manifest."""
    problems = lint.check(fig)
    errors = [p for p in problems if p.severity == "error"]
    if errors:
        raise AssertionError(f"{visual_id}: " + "; ".join(str(e) for e in errors))
    for suffix in ("png", "svg"):
        path = VERIFY_DIR / f"{visual_id}.{suffix}"
        fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=150)
    # matplotlib emits trailing spaces inside SVG path data, which shows up as a
    # dirty `git diff --check` on every re-run. Strip it here rather than by hand.
    svg_path = VERIFY_DIR / f"{visual_id}.svg"
    svg_path.write_text(
        "\n".join(line.rstrip() for line in svg_path.read_text(encoding="utf-8").splitlines()) + "\n",
        encoding="utf-8",
    )
    VERIFY_MANIFEST.append({
        "visual_id": visual_id,
        "headline": headline,
        "png": f"outputs/report_visuals/verification/{visual_id}.png",
        "svg": f"outputs/report_visuals/verification/{visual_id}.svg",
        "source": "scripts/metrics.py (certified base) vs outputs/sales_master.csv (raw base)",
        "caveat": caveat,
        "lint_warnings": "; ".join(str(p) for p in problems) or "none",
    })
    display(fig)
    plt.close(fig)


# --- the exact checks -----------------------------------------------------
pnl_total = CERTIFIED.pnl_metrics_df.query("brand.str.lower() == 'total'").iloc[0]
scc_prior = pnl_total.prior_supply_chain_cost_eur / pnl_total.prior_turnover_eur
scc_current = pnl_total.current_supply_chain_cost_eur / pnl_total.current_turnover_eur

EXACT_CHECKS = [
    ("2.1 raw GSV FY2023", 371528.3, raw_rows.query("y == @FY_PRIOR").gsv_keur.sum(), 0.05),
    ("2.1 raw GSV FY2024", 349247.6, raw_rows.query("y == @FY_CURRENT").gsv_keur.sum(), 0.05),
    ("2.1 raw TO FY2023", 290066.9, raw_rows.query("y == @FY_PRIOR").turnover_keur.sum(), 0.05),
    ("2.1 raw TO FY2024", 270504.3, raw_rows.query("y == @FY_CURRENT").turnover_keur.sum(), 0.05),
    ("2.1 raw Discount FY2023", 81461.4, raw_rows.query("y == @FY_PRIOR").discount_keur.sum(), 0.05),
    ("2.1 raw Discount FY2024", 78743.3, raw_rows.query("y == @FY_CURRENT").discount_keur.sum(), 0.05),
    ("2.1 TO>GSV rows FY2023 TO", 2574.5,
     _to(raw_rows, FY_PRIOR) - _to(cert_rows, FY_PRIOR), 0.05),
    ("2.1 TO>GSV rows FY2024 TO", 3173.0,
     _to(raw_rows, FY_CURRENT) - _to(cert_rows, FY_CURRENT), 0.05),
    ("4.6 Gross Profit FY2023", 138723.9, pnl_total.prior_gross_profit_eur, 0.1),
    ("4.6 Gross Profit FY2024", 128421.7, pnl_total.current_gross_profit_eur, 0.1),
    ("4.6 Gross Profit change", -10302.1, pnl_total.gross_profit_change_eur, 0.1),
    ("4.6 Gross Margin FY2023 %", 47.82, pnl_total.prior_gross_margin_pct * 100, 0.01),
    ("4.6 Gross Margin FY2024 %", 47.47, pnl_total.current_gross_margin_pct * 100, 0.01),
    ("4.6 Gross Margin movement bps", -35, pnl_total.gross_margin_movement_bps, 0.6),
    ("4.6 Marketing FY2023", 58000.0, pnl_total.prior_marketing_expense_eur, 0.1),
    ("4.6 Marketing FY2024", 42500.0, pnl_total.current_marketing_expense_eur, 0.1),
    ("4.6 Marketing change", -15500.0, pnl_total.marketing_expense_change_eur, 0.1),
    ("4.6 PBO FY2023", 80723.9, pnl_total.prior_pbo_eur, 0.1),
    ("4.6 PBO FY2024", 85921.7, pnl_total.current_pbo_eur, 0.1),
    ("4.6 PBO change", 5197.9, pnl_total.pbo_change_eur, 0.1),
    ("4.6 PBO margin FY2023 %", 27.83, pnl_total.prior_pbo_margin_pct * 100, 0.01),
    ("4.6 PBO margin FY2024 %", 31.76, pnl_total.current_pbo_margin_pct * 100, 0.01),
    ("4.6 PBO margin movement bps", 393, pnl_total.pbo_margin_movement_bps, 1.0),
    ("4.6 PBO bridge residual", 0.0, pnl_total.pbo_bridge_check_eur, 1e-6),
    ("4.6 Supply chain cost change", -9260.0, pnl_total.supply_chain_cost_change_eur, 15),
    ("4.6 Supply Chain Cost/TO FY2023 %", 52.18, scc_prior * 100, 0.01),
    ("4.6 Supply Chain Cost/TO FY2024 %", 52.53, scc_current * 100, 0.01),
    ("4.6 Supply Chain Cost/TO movement bps", 35, (scc_current - scc_prior) * 10_000, 0.6),
]

_market = CERTIFIED.market_metrics_df


def _mkt(channel, brand=None, manufacturer=None):
    rows = _market[_market["channel"].eq(channel)]
    rows = rows[rows["brand"].isna()] if brand is None else rows[rows["brand"].eq(brand)]
    if manufacturer:
        rows = rows[rows["manufacturer"].eq(manufacturer)]
    return rows.iloc[0]


EXACT_CHECKS += [
    ("4.7 Category growth %", -1.3, _mkt("Total", manufacturer="Category").market_value_growth_pct * 100, 0.05),
    ("4.7 UniSweet growth %", -4.3, _mkt("Total", manufacturer="UNISWEET").market_value_growth_pct * 100, 0.05),
    ("4.7 UniSweet share movement pp", -1.97, _mkt("Total", manufacturer="UNISWEET").share_movement_pp, 0.01),
    ("4.7 OLIVE share movement pp", -2.98, _mkt("Total", brand="OLIVE").share_movement_pp, 0.01),
    ("4.7 SKY share movement pp", 0.45, _mkt("Total", brand="SKY").share_movement_pp, 0.01),
    ("4.7 COBALT share movement pp", 0.56, _mkt("Total", brand="COBALT").share_movement_pp, 0.01),
    ("4.7 DT UniSweet share movement pp", -0.28, _mkt("DT", manufacturer="UNISWEET").share_movement_pp, 0.01),
    ("4.7 MT UniSweet share movement pp", -6.07, _mkt("MT", manufacturer="UNISWEET").share_movement_pp, 0.01),
    ("4.7 NAVY DT share movement pp", 0.84, _mkt("DT", brand="NAVY").share_movement_pp, 0.01),
    ("4.7 LILAC MT share movement pp", 5.53, _mkt("MT", brand="LILAC").share_movement_pp, 0.01),
    ("4.7 OLIVE-MT share movement pp", -7.88, _mkt("MT", brand="OLIVE").share_movement_pp, 0.01),
]

verification = pd.DataFrame(
    [{"check": name, "framework": stated, "computed": actual,
      "difference": actual - stated, "status": "PASS" if abs(actual - stated) <= tol else "FAIL"}
     for name, stated, actual, tol in EXACT_CHECKS]
)
CHECKS_PASSED = int((verification["status"] == "PASS").sum())
CHECKS_TOTAL = len(verification)

market_recon_ok = (
    int((_market["sales_value_gain_loss_matches_source"] == False).sum()) == 0
    and int((_market["share_gain_loss_matches_source"] == False).sum()) == 0
)
assert market_recon_ok, "Market source gain/loss columns no longer reconcile"
assert CHECKS_PASSED == CHECKS_TOTAL, verification.query("status == 'FAIL'").to_string()

print(f"{CHECKS_PASSED}/{CHECKS_TOTAL} exact checks pass; "
      f"market source gain/loss reconciles on all rows")
display(verification.style.format({"framework": "{:,.2f}", "computed": "{:,.2f}",
                                   "difference": "{:+,.4f}"}).hide(axis="index"))

39/39 exact checks pass; market source gain/loss reconciles on all rows


check,framework,computed,difference,status
2.1 raw GSV FY2023,"371,528.30","371,528.30",+0.0000,PASS
2.1 raw GSV FY2024,"349,247.60","349,247.60",+0.0000,PASS
2.1 raw TO FY2023,"290,066.90","290,066.90",+0.0000,PASS
2.1 raw TO FY2024,"270,504.30","270,504.30",+0.0000,PASS
2.1 raw Discount FY2023,"81,461.40","81,461.40",+0.0000,PASS
2.1 raw Discount FY2024,"78,743.30","78,743.30",-0.0000,PASS
2.1 TO>GSV rows FY2023 TO,"2,574.50","2,574.50",+0.0000,PASS
2.1 TO>GSV rows FY2024 TO,"3,173.00","3,173.00",+0.0000,PASS
4.6 Gross Profit FY2023,"138,723.90","138,723.85",-0.0450,PASS
4.6 Gross Profit FY2024,"128,421.70","128,421.75",+0.0450,PASS


In [11]:
# V1 - verification result as a single number
fig, ax = swd.figure(
    "Every framework figure reconciles on the base it is assigned to",
    subtitle="Exact checks of STORYLINE_METRIC_FRAMEWORK.md against scripts/metrics.py",
    source=("Checks: raw-base reconciliation, P&L levels, margins, PBO bridge, supply-chain rate, "
            "Market growth and share  |  Market source gain/loss ties on every row"),
    figsize=(13.3, 5.6))
swd.bignum(ax, f"{CHECKS_PASSED} / {CHECKS_TOTAL}",
           "exact checks pass - the certified engine's formulas match section 4 of the framework")
save_verification(
    fig, "V1_verification_result",
    "Every framework figure reconciles on the base it is assigned to",
    "Confirms formula compliance and reconciliation, not that the raw base is validated; "
    "TO > GSV rows remain open")

<Figure size 1995x840 with 1 Axes>

In [12]:
# V2 - would the certified base change the growth rates the storyline quotes?
# OLIVE (-7.3%) and SKY (-7.4%) sit 0.1pp apart and within 0.7pp of Total; on a
# slopegraph their labels overlap illegibly, so they are quoted in the footnote.
cuts = [("Total", None, None), ("COBALT", "brand_name", "COBALT"),
        ("DT", "channel_code", "DT"), ("MT", "channel_code", "MT")]
growth_pairs = {label: (to_growth(raw_rows, col, key), to_growth(cert_rows, col, key))
                for label, col, key in cuts}

fig, ax = swd.figure(
    "The certified base makes every decline look steeper, most of all in MT",
    subtitle=f"FY{FY_CURRENT} turnover growth %, same formula on two bases",
    source=("Source: scripts/metrics.py  |  Certified base excludes rows flagged TO > GSV "
            "(2,574.5 kEUR of FY2023 TO, 3,173.0 kEUR of FY2024 TO). "
            "OLIVE -7.3% -> -7.5% and SKY -7.4% -> -7.6% omitted: they overlap Total"))
swd.slopegraph(ax, "Raw / P&L-aligned\n(storyline base)", "Certified\n(metrics.py base)",
               growth_pairs, highlight={"MT", "COBALT"}, unit="%", decimals=1)
save_verification(
    fig, "V2_growth_base_sensitivity",
    "The certified base makes every decline look steeper, most of all in MT",
    "Two different bases, not two periods; the framework quotes the raw base throughout")

<Figure size 1920x1080 with 1 Axes>

In [13]:
# V3 - which drivers actually move when the base changes
entities = [("Macarons", "customer_name"), ("MT", "channel_code"), ("OLIVE", "brand_name"),
            ("PACK 1.1KG", "product_name"), ("COBALT", "brand_name"),
            ("POUCH 900GR", "product_name"), ("POUCH 400GR", "product_name"),
            ("Candies", "customer_name"), ("Bliss", "customer_name"),
            ("DT", "channel_code"), ("POUCH 100GR", "product_name")]
base_gap = [(name, (to_change(cert_rows, col, name) - to_change(raw_rows, col, name)) / 1000)
            for name, col in entities]

fig, ax = swd.figure(
    "Macarons is the one driver whose size changes materially with the base",
    subtitle=("Certified minus raw FY2024 turnover change, EURm "
              "(negative = the certified base shows a bigger loss)"),
    source=("Source: scripts/metrics.py  |  Sensitivity, not a correction. On the certified base "
            "Macarons reads -5.60m / -35.7% instead of -4.35m / -25.7%"))
swd.hbar(ax, [row[0] for row in base_gap], [row[1] for row in base_gap],
         highlight={"Macarons", "MT"}, decimals=2, sort=True)
save_verification(
    fig, "V3_driver_base_gap",
    "Macarons is the one driver whose size changes materially with the base",
    "Framework guardrail 5 still holds on both bases: this is concentration, not attrition")

<Figure size 1920x1080 with 1 Axes>

In [14]:
# V4 - why H2 concentration is the most base-sensitive claim
flagged = raw_rows[~raw_rows["certified_for_analysis"]]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
flagged_by_month = []
for month in range(1, 13):
    subset = flagged[flagged["reporting_month"].dt.month == month]
    flagged_by_month.append(
        (subset.loc[subset["y"] == FY_CURRENT, "turnover_keur"].sum()
         - subset.loc[subset["y"] == FY_PRIOR, "turnover_keur"].sum()) / 1000)

h1_effect, h2_effect = sum(flagged_by_month[:6]), sum(flagged_by_month[6:])

fig, ax = swd.figure(
    f"Flagged rows add {h1_effect:.1f}m to H1 and take {abs(h2_effect):.1f}m off H2, "
    f"softening the H2 story",
    subtitle="FY2024 vs FY2023 turnover change contributed by rows excluded from the certified base, EURm",
    source=("Source: scripts/metrics.py  |  Positive = the flagged rows make that month look better "
            "on the raw base. H2 share of the FY decline is 95.7% raw, 90.2% certified"))
swd.bar(ax, month_names, flagged_by_month, highlight={"Mar", "Jul", "Dec"}, decimals=2)
save_verification(
    fig, "V4_flagged_rows_by_month",
    "Flagged rows flatter H1 and depress H2, softening the H2 concentration claim",
    "Framework guardrail 6 still applies: monthly sell-in does not establish seasonal causality")

<Figure size 1920x1080 with 1 Axes>

In [15]:
# --- Appendix export manifest and rcParams restore ------------------------
verify_manifest = pd.DataFrame(VERIFY_MANIFEST)
verify_manifest.to_csv(VERIFY_DIR / "verification_manifest.csv", index=False)
verification.to_csv(VERIFY_DIR / "framework_check_results.csv", index=False)

for row in VERIFY_MANIFEST:
    png = PROJECT_ROOT / row["png"]
    with Image.open(png) as image:
        assert image.width > 0 and image.height > 0, png
    svg = PROJECT_ROOT / row["svg"]
    tree = ElementTree.parse(svg)
    assert "Arial" in svg.read_text(encoding="utf-8"), f"{svg} lost the Arial font stack"

plt.rcParams.update(_PROJECT_RCPARAMS)   # hand the project palette back

print(f"{len(VERIFY_MANIFEST)} verification visuals exported to {VERIFY_DIR}")
print(f"lint: {sum(1 for r in VERIFY_MANIFEST if r['lint_warnings'] == 'none')}"
      f"/{len(VERIFY_MANIFEST)} clean")
display(verify_manifest[["visual_id", "headline", "lint_warnings"]])

4 verification visuals exported to /Users/tranvomanhtuan/Documents/05_Business_Case_Studies/SEO-V FBP Case Data/outputs/report_visuals/verification
lint: 4/4 clean


,visual_id,headline,lint_warnings
0,V1_verification_result,Every framework figure reconciles on the base ...,none
1,V2_growth_base_sensitivity,The certified base makes every decline look st...,none
2,V3_driver_base_gap,Macarons is the one driver whose size changes ...,none
3,V4_flagged_rows_by_month,"Flagged rows flatter H1 and depress H2, soften...",none
